In [1]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import torch.nn as nn

In [2]:
df = pd.read_csv("../Veriler/08-seismic_activity_svm.csv")
df.head()

,underground_wave_energy,vibration_axis_variation,seismic_event_detected
0,9.539392,-3.000000,0
1,9.558241,-2.939394,0
2,9.576669,-2.878788,0
3,9.594678,-2.818182,0
4,9.612272,-2.757576,0


In [3]:
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 3 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   underground_wave_energy   400 non-null    float64
 1   vibration_axis_variation  400 non-null    float64
 2   seismic_event_detected    400 non-null    int64  
dtypes: float64(2), int64(1)
memory usage: 9.5 KB
None
       underground_wave_energy  vibration_axis_variation  \
count               400.000000              4.000000e+02   
mean                  0.000000              8.881784e-18   
std                   7.719350              1.751650e+00   
min                  -9.999954             -3.000000e+00   
25%                  -6.134779             -1.500000e+00   
50%                   0.000000              0.000000e+00   
75%                   6.134779              1.500000e+00   
max                   9.999954              3.000000e+00   

       seismic_event_dete

In [4]:
x = torch.tensor(df[['underground_wave_energy','vibration_axis_variation']].values,dtype=torch.float32)
y = torch.tensor(df['seismic_event_detected'].values,dtype=torch.float32).unsqueeze(1)

In [5]:
x_train, x_test, y_train, y_test = train_test_split(x,y, train_size=0.8, random_state=42)

In [44]:
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=2,out_features=5,dtype=torch.float32)
        self.relu_1 = nn.ReLU()
        self.layer_2 = nn.Linear(in_features=5,out_features=7,dtype=torch.float32)
        self.relu_2 = nn.ReLU()
        self.layer_3 = nn.Linear(in_features=7,out_features=1,dtype=torch.float32)
        
    def forward(self,x):
        out = self.layer_1(x)
        out = self.relu_1(out)
        out = self.layer_2(out)
        out = self.relu_2(out)
        out = self.layer_3(out)
        return out

In [45]:
model = SimpleModel()

In [46]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(params = model.parameters(), lr = 0.001)

In [47]:
def calculate_accuracy(y_true, y_pred):
    correct = torch.eq(y_true,y_pred).sum().item()
    accuracy = (correct / len(y_pred)) *100
    return accuracy

In [48]:
torch.manual_seed(42)
epochs = 200

for epoch in range(epochs):
    model.train()
    y_logits = model(x_train)
    y_pred = torch.round(torch.sigmoid(y_logits))
    loss = loss_fn(y_logits,y_train)
    acc = calculate_accuracy(y_train,y_pred)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.inference_mode():
        test_logits = model(x_test)
        test_pred = torch.round(torch.sigmoid(test_logits))

        test_loss = loss_fn(test_logits,y_test)
        test_acc  = calculate_accuracy(y_test,test_pred) 

        if epoch % 10 == 0:
            print(f"Epoch:{epoch}, Loss:{loss}, Acc:{acc}, Test_Loss:{test_loss}, Test_Acc:{test_acc}")

Epoch:0, Loss:0.6694039106369019, Acc:60.62499999999999, Test_Loss:0.674588680267334, Test_Acc:62.5
Epoch:10, Loss:0.6560729146003723, Acc:65.625, Test_Loss:0.6572448015213013, Test_Acc:68.75
Epoch:20, Loss:0.6436833739280701, Acc:72.5, Test_Loss:0.6411381959915161, Test_Acc:75.0
Epoch:30, Loss:0.632324755191803, Acc:72.8125, Test_Loss:0.6270648241043091, Test_Acc:73.75
Epoch:40, Loss:0.621374249458313, Acc:72.5, Test_Loss:0.6142783164978027, Test_Acc:71.25
Epoch:50, Loss:0.6103253960609436, Acc:72.5, Test_Loss:0.6016790866851807, Test_Acc:70.0
Epoch:60, Loss:0.5992022156715393, Acc:72.8125, Test_Loss:0.5896022319793701, Test_Acc:70.0
Epoch:70, Loss:0.5876861810684204, Acc:73.4375, Test_Loss:0.5779560804367065, Test_Acc:71.25
Epoch:80, Loss:0.5757625102996826, Acc:74.6875, Test_Loss:0.5663569569587708, Test_Acc:73.75
Epoch:90, Loss:0.5631799697875977, Acc:75.9375, Test_Loss:0.5537682771682739, Test_Acc:73.75
Epoch:100, Loss:0.5502765774726868, Acc:76.25, Test_Loss:0.5414308309555054, T